# Can we predict next-month arrears?
Target: `arrears_next_month`. The model is meant to score future billing months, so the split is **temporal**; accounts repeat across months, so CV is also forward-chaining by month.

**Feature audit (checklist 7).** `meter_ref` is an identifier -> dropped. `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. not known at prediction time (and it is nearly a proxy for the outcome: 87% vs 1.4% arrears) -> dropped. `billing_month` is used only for splitting. Kept: `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df = pd.read_csv('data.csv', parse_dates=['billing_month']).sort_values('billing_month').reset_index(drop=True)
print(df.shape, df.billing_month.min().date(), df.billing_month.max().date(), 'prevalence', df.arrears_next_month.mean().round(3))

## 1. Test set, set aside before any modelling
Latest 6 billing months (2025-07 .. 2025-12) are the test set. 2025-06 is dropped as a one-month gap, since its outcome (arrears in July) overlaps the first test month. Everything else is training.

In [ ]:
TARGET='arrears_next_month'
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']   # leak/identifier columns excluded, see above
test_mask = df.billing_month >= '2025-07-01'
gap_mask = df.billing_month == '2025-06-01'
train, test = df[~test_mask & ~gap_mask].copy(), df[test_mask].copy()
print(len(train), len(test), 'train prevalence', train[TARGET].mean().round(3), 'test prevalence', test[TARGET].mean().round(3))
print('accounts in test also seen in train:', test.meter_ref.isin(train.meter_ref).mean().round(3))

## 2. Candidate models (named before any scoring)
1. `baseline_prior` - constant prediction of the training prevalence.
2. `logreg` - one-hot tariff + standardised numerics, logistic regression, C in {0.01, 0.1, 1, 10}.
3. `hgb` - histogram gradient boosting, (max_depth, learning_rate) in {2,4} x {0.05, 0.1}, 150 iterations.

Selection metric: mean PR-AUC over forward-chaining CV folds (positive class is ~12%). Decision threshold: maximise F1 on pooled out-of-fold predictions of the chosen model. All preprocessing lives inside the pipelines, so it is fit within each fold.

In [ ]:
def prep():
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['tariff']),
                              ('num', StandardScaler(), ['occupants','kwh','bill_eur','late_payments_12m'])])
def make(name, **p):
    if name=='baseline_prior': return DummyClassifier(strategy='prior')
    if name=='logreg': return make_pipeline(prep(), LogisticRegression(C=p['C'], max_iter=1000))
    if name=='hgb':
        pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['tariff'])], remainder='passthrough')
        return make_pipeline(pre, HistGradientBoostingClassifier(max_depth=p['max_depth'], learning_rate=p['learning_rate'], max_iter=150, random_state=0))
CANDIDATES = [('baseline_prior',{})] + [('logreg',{'C':c}) for c in (0.01,0.1,1,10)] + \
             [('hgb',{'max_depth':d,'learning_rate':l}) for d in (2,4) for l in (0.05,0.1)]

## 3. Forward-chaining cross-validation inside the training rows
Five folds: train on all months up to a cut-off, validate on the next 3 months after a 1-month gap.

In [ ]:
months = np.sort(train.billing_month.unique())
folds=[]
for k in range(5):
    val_end = len(months) - 3*(4-k); val_start = val_end-3
    tr = train.billing_month.isin(months[:val_start-1]); va = train.billing_month.isin(months[val_start:val_end])
    folds.append((np.where(tr)[0], np.where(va)[0]))
    print(k, 'train through', months[val_start-2].astype('datetime64[M]'), '| val', months[val_start].astype('datetime64[M]'), '..', months[val_end-1].astype('datetime64[M]'), len(folds[-1][0]), len(folds[-1][1]))
Xtr, ytr = train[FEATURES], train[TARGET].values
rows=[]; oof={}
for name,p in CANDIDATES:
    pred=np.full(len(train), np.nan); aps=[]; aucs=[]
    for tr,va in folds:
        m=make(name,**p).fit(Xtr.iloc[tr], ytr[tr]); s=m.predict_proba(Xtr.iloc[va])[:,1]; pred[va]=s
        aps.append(average_precision_score(ytr[va],s)); aucs.append(roc_auc_score(ytr[va],s))
    key=name+str(p); oof[key]=pred
    rows.append(dict(model=name, params=p, cv_pr_auc=np.mean(aps), cv_pr_auc_sd=np.std(aps), cv_roc_auc=np.mean(aucs), key=key))
cv=pd.DataFrame(rows).sort_values('cv_pr_auc',ascending=False); print(cv.drop(columns='key').to_string())

In [ ]:
best=cv.iloc[0]; mask=~np.isnan(oof[best.key])
grid=np.linspace(0.02,0.95,94)
f1s=[f1_score(ytr[mask], oof[best.key][mask]>=t) for t in grid]
thr=grid[int(np.argmax(f1s))]
print(best.model, best.params, '| CV-chosen threshold', round(thr,2))

## 4. Choice, recorded from cross-validation only (before the test set is touched)
- **Model:** `logreg` with C = 0.1 (CV PR-AUC 0.161 vs 0.054 for the prior baseline; best of all 9 candidates, ahead of every gradient-boosting setting).
- **Features:** `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.
- **Decision threshold:** 0.07 predicted probability (maximises F1 on pooled out-of-fold predictions).
- Caveat seen in CV: fold-to-fold PR-AUC spread is large (sd ~0.13) and arrears prevalence rises over time, so scores are expected to be noisy and the threshold may not transfer.

Next cell refits on all training rows and scores the test set once. No further changes follow.

In [ ]:
final = make('logreg', C=0.1).fit(train[FEATURES], ytr)
s = final.predict_proba(test[FEATURES])[:,1]; yte = test[TARGET].values; yhat = s >= 0.07
base = make('baseline_prior').fit(train[FEATURES], ytr).predict_proba(test[FEATURES])[:,1]
print('TEST  (headline) ROC-AUC', round(roc_auc_score(yte,s),3), '| PR-AUC', round(average_precision_score(yte,s),3),
      '| precision', round(precision_score(yte,yhat),3), '| recall', round(recall_score(yte,yhat),3), '| F1', round(f1_score(yte,yhat),3))
print('prior baseline PR-AUC (= test prevalence level)', round(average_precision_score(yte,base),3), '| test prevalence', round(yte.mean(),3))

## 5. Reading the result
See the printed test line above as the single headline score. Whether it is useful depends on how it compares with the baseline PR-AUC (about the test prevalence) and on the prevalence shift between training and test periods, which the CV threshold did not anticipate. This is a one-shot estimate on 6 months; it is not tuned further.